In [28]:
import torch
import matplotlib.pyplot as plt

from model import SimpleSegmentationCNN
from dataset import CerebellumSliceDataset
from torch.utils.data import DataLoader
from utils import get_device, calculate_dice_score
import numpy as np
import nibabel as nib

In [29]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
checkpoint = "best_cerebellum_model.pth"
model = SimpleSegmentationCNN().to(device)

checkpoint = torch.load(checkpoint, map_location=device)
model.load_state_dict(checkpoint["state_dict"])

print(f"Loaded best model from Epoch {checkpoint["epoch"]} with Validation Dice: {checkpoint["val_dice"]:.4f}")

Loaded best model from Epoch 50 with Validation Dice: 0.5209


In [30]:
model.eval()
volume_predictions = {}
all_predictions = []
all_targets = []
all_images = []

test_dataset = CerebellumSliceDataset(dataset_split="test")
test_loader = DataLoader(dataset=test_dataset, batch_size=4, shuffle=False, num_workers=0)

total_tp = 0.0
total_fp = 0.0
total_fn = 0.0
smooth = 1e-6

with torch.no_grad():
    for images, masks, patients, filenames in test_loader:
        images = images.to(device)
        masks = masks.to(device)
        
        predictions = model(images)
        
        binary_predictions = (predictions > 0.5).float()
        binary_targets = (masks > 0.5).float()
        
        flat_predictions = binary_predictions.view(-1)
        flat_targets = binary_targets.view(-1)
        
        total_tp += (flat_predictions * flat_targets).sum().item()
        total_fp += (flat_predictions * (1.0 - flat_targets)).sum().item()
        total_fn += ((1.0 - flat_predictions) * flat_targets).sum().item()
         
        preds_np = binary_predictions.cpu().squeeze(1).numpy()
         
        for i, patient_id in enumerate(patients):
            pid = patient_id if isinstance(patient_id, str) else patient_id[i]
            if pid not in volume_predictions:
                volume_predictions[pid] = []
            volume_predictions[pid].append(preds_np[i])
             
        all_images.extend(images.cpu())
        all_targets.extend(masks.cpu())
        all_predictions.extend(predictions.cpu())

dice = (2.0 * total_tp + smooth) / (2.0 * total_tp + total_fp + total_fn + smooth)
iou = (total_tp + smooth) / (total_tp + total_fp + total_fn + smooth)
precision = (total_tp + smooth) / (total_tp + total_fp + smooth)
recall = (total_tp + smooth) / (total_tp + total_fn + smooth)

print("================ FINAL TEST SET RESULTS ================")
print(f"Dice Score:    {dice:.4f}")
print(f"IoU (Jaccard): {iou:.4f}")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")
print("========================================================")

================ FINAL TEST SET RESULTS ================
Dice Score:    0.6834
IoU (Jaccard): 0.5191
Precision:     0.6057
Recall:        0.7840


In [31]:
import os

In [32]:
# output_dir = "simple_cnn_nifti_outputs"
# os.makedirs(output_dir, exist_ok=True)

# print(f"\nExporting patient volumes to '{output_dir}'...")
# for pid, slice_list in volume_predictions.items():
#     volume_3d = np.stack(slice_list, axis=-1).astype(np.float32) # (H, W, Z)
#     affine = np.eye(4)
    
#     pred_nii = nib.Nifti1Image(volume_3d, affine)
#     output_filename = os.path.join(output_dir, f"{pid}_simple_cnn_pred.nii.gz")
#     nib.save(pred_nii, output_filename)
#     print(f"Saved: {output_filename} with shape {volume_3d.shape}")

In [33]:
# plotted = 0
# for i in range(len(all_targets)):
#     if all_targets[i].sum().item() > 0:
#         img = all_images[i].squeeze()
#         target = all_targets[i].squeeze()
#         pred_prob = all_predictions[i].squeeze()
#         pred_mask = (pred_prob > 0.5).float()
        
#         fig, axes = plt.subplots(1, 3, figsize=(12, 4))
#         axes[0].imshow(img, cmap="gray")
#         axes[0].set_title(f"Slice {i}: Original MRI")
#         axes[0].axis("off")
        
#         axes[1].imshow(target, cmap="gray")
#         axes[1].set_title("Ground Truth Mask")
#         axes[1].axis("off")
        
#         axes[2].imshow(pred_mask, cmap="gray")
#         axes[2].set_title("Simple CNN Prediction")
#         axes[2].axis("off")
        
#         plt.tight_layout()
#         plt.show()
        
#         plotted += 1
#         if plotted >= 4:
#             break

In [34]:
target_filenames = [
    ("patient4", "OASIS-TRT-20-4_coronal-059.png"),
    ("patient4", "OASIS-TRT-20-4_coronal-105.png"),
    ("patient1", "OASIS-TRT-20-1_coronal-039.png"), 
    ("patient1", "OASIS-TRT-20-1_coronal-094.png")
]

In [35]:
import cv2

In [ ]:
model.eval()

with torch.no_grad():
    for folder, file in target_filenames:
        img_path = os.path.join("../data_processed", "images", "test", folder, file)
        mask_path = os.path.join("../data_processed", "masks", "test", folder, file)
        
        raw_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        raw_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        img_norm = raw_img.astype(np.float32) / 255.0
        mask_norm = raw_mask.astype(np.float32) / 255.0
        
        img_tensor = torch.from_numpy(img_norm).unsqueeze(0).unsqueeze(0).to(device)
        
        mask_tensor = torch.from_numpy(mask_norm).to(device)
        mask_tensor = (mask_tensor > 0.5).float().view(-1)
        
        pred_logits = model(img_tensor)
        pred_binary = (pred_logits > 0.5).float().view(-1)
        
        tp = (pred_binary * mask_tensor).sum().item()
        fp = (pred_binary * (1.0 - mask_tensor)).sum().item()
        fn = ((1.0 - pred_binary) * mask_tensor).sum().item()
        
        dice_score = (2.0 * tp + smooth) / (2.0 * tp + fp + fn + smooth)
        print(f"{folder} | {file} | Dice: {dice_score:.4f}")
        
        pred_2d = pred_binary.view(raw_img.shape).cpu().numpy()
        
        pred_img_to_save = (pred_2d * 255).astype(np.uint8)
        
        save_dir = os.path.join(".", "test_cases", "test", folder)
        os.makedirs(save_dir, exist_ok=True)
        
        # 4. Write the file to disk 
        save_path = os.path.join(save_dir, f"pred_{file}")
        cv2.imwrite(save_path, pred_img_to_save)

patient4 | OASIS-TRT-20-4_coronal-059.png | Dice: 0.8872
patient4 | OASIS-TRT-20-4_coronal-105.png | Dice: 0.1429
patient1 | OASIS-TRT-20-1_coronal-039.png | Dice: 0.0000
patient1 | OASIS-TRT-20-1_coronal-094.png | Dice: 0.6481


In [ ]:
# Measures how closely the predicted cerebellum mask matches the true mask.
# It is a good main metric because it focuses on the masks, not just accuracy, which would
# be skewed by the many background image pixels.

In [ ]:
# Also measures the overlap between
# the predicted and true masks, but is stricter than dice because unlike dice, the overlapping
# pixels only show up once in the numerator.

In [ ]:
# Measures how much of the real cerebellum the model finds. High recall
# means the model is less likely to miss parts of the cerebellar anatomy.

In [ ]:
# Measures how much of the real cerebellum the model finds. High recall
# means the model is less likely to miss parts of the cerebellar anatomy.

In [ ]:
# Measures whether the pixels predicted as cerebellum are actually
# cerebellum. High precision means the model is less likely to incorrectly include nearby
# brain tissue